In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
from supabase import create_client, Client
supabase_url = os.environ.get("SUPABASE_URL")
supabase_api_key = os.environ.get("SUPBASE_KEY")

supabase: Client = create_client(supabase_url, supabase_api_key)

In [2]:
from openai import OpenAI
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [6]:
def workflow_research(date, next_date):
    result = supabase.table('curation_selected_items')\
        .select('event_id, output,sources, news_date','topic')\
    .gte('created_at', date)\
    .lt('created_at', next_date)\
    .eq('topic', 'Workflow improvement')\
        .order('created_at', desc=False)\
        .execute()
    
    return result.data

In [17]:
workflow_deep_dive = workflow_research('2026-02-26','2026-02-27')

In [18]:
workflow_deep_dive

[{'event_id': '1529_2026-02-24',
  'output': 'Notion 3.3: Custom Agents released (Feb 24, 2026). Key points: fully autonomous agents for team workflows (triage, Q&A, scheduled reports, task routing), built-in templates and multi-tool integrations (Slack, Figma, Mail, Calendar, Linear, HubSpot, FigJam). Notion reports early high adoption (thousands of agents built), admin-level controls and logging, and a consumption-based cost model: agents consume Notion credits (free Business trial until 3 May 2026; pay-for-work pricing begins 4 May 2026). Practical impact: product, marketing and design teams can automate recurring workflows, synthesize cross-tool context, and embed agentic automation into content, feedback, and reporting workflows.',
  'sources': [{'url': 'https://www.notion.com/en-gb/releases/2026-02-24',
    'name': 'Notion'}],
  'news_date': '2026-02-24',
  'topic': 'Workflow improvement'},
 {'event_id': '1548_2026-02-24',
  'output': 'Canva announced acquisitions of MangoAI and 

In [19]:
def openai_research_workflow(research_input):
    for i in research_input:
        response = client.responses.create(
            model = "gpt-5-nano",
            tools = [{
                "type": "web_search"
            }],
            include = ["web_search_call.action.sources"],
            input = f"""
            You are a senior research analyst for Krux.

            TASK:
              Create a deep, fact-checked brief for ONE AI report/paper/benchmark release.
            
            GOAL:
              Extract the highest-value findings and implementation guidance so a later 100-word summary can capture most practical signal.
            
            OUTPUT FORMAT:
              - 12 to 16 bullet points.
              - Each bullet must include inline citations:
                [Source: <publisher>, <url>]
              - No text before or after bullets.
            
            MANDATORY STRUCTURE:
              1) REPORT SNAPSHOT
              - What was published, by whom, and when.
              - Scope: geography, sectors, sample size, time window, method type.
            
              2) CORE FINDINGS (MOST IMPORTANT)
              - 4 to 6 bullets with the strongest quantified findings.
              - Include exact numbers, not vague language.
            
              3) WHAT THIS ACTUALLY MEANS
              - 2 to 3 bullets translating findings into plain English implications for real teams.
            
              4) IMPLEMENTATION PLAYBOOK
              - 3 to 4 bullets: what organizations should do in the next quarter.
              - Include sequencing (e.g., data readiness -> pilot -> measurement -> rollout).
            
              5) METRICS TO TRACK
              - 1 or 2 bullets on KPIs teams should monitor to apply the report in practice.
            
              6) LIMITATIONS / CAVEATS
              - 1 or 2 bullets on methodology limits, sample bias, missing data, or uncertainty.
            
              7) DECISION TAKEAWAY
              - 1 bullet: “If a team only does one thing based on this report, it should be X.”
            
              STRICT RULES:
              - No hype, no opinion, no generic AI commentary.
              - Every factual claim must be source-backed.
              - If a key detail is missing, write: "Not disclosed in cited sources."
              - Prefer primary sources (original report/paper) over secondary articles.
              - If secondary coverage conflicts with primary report, prioritize primary and note conflict.
            
              QUALITY CHECK:
              - Must contain quantified findings.
              - Must contain actionable implementation steps.
              - Must clearly separate findings from interpretation.
              - Must include limitations.
            
              Report/event to research:
              {i['output']}
            """,
        )

        output = response.output_text
        print(output)

        final_dictionary = {
            'event_id': i['event_id'],
            'news_date': i['news_date'],
            'output': output,
            'model_provider': 'openai',
            'topic': i['topic']
        }
        print(final_dictionary)

        save_research(final_dictionary)

        print("✅ Done")

In [20]:
def save_research(research_json):
    supabase.table('research_assistant').insert({
        'event_id': research_json['event_id'],
        'model_provider': research_json['model_provider'],
        'news_date': research_json['news_date'],
        'output': research_json['output'],
        'topic': research_json['topic']
    }).execute()

In [21]:
openai_research_workflow(workflow_deep_dive[0:])

- REPORT SNAPSHOT: What was published, by whom, and when — Notion released Notion 3.3: Custom Agents on February 24, 2026, published by Notion HQ (official release page). [Source: Notion, https://www.notion.com/releases]

- REPORT SNAPSHOT: Scope and context — Global Notion workspace feature aimed at Business and Enterprise teams; exact geographic reach and sample size not disclosed; time window centers on a public beta starting Feb 24, 2026 with a free window through May 3, 2026; method is a product release with autonomous agents and templates integrated into workflows. [Source: Notion, https://www.notion.com/releases]

- CORE FINDINGS (1) Autonomy and uptime — Custom Agents operate autonomously (no manual prompting) and can run 24/7 once configured with triggers or schedules. [Source: Notion, https://www.notion.com/blog/introducing-custom-agents]

- CORE FINDINGS (2) Team-oriented design and governance — Agents are designed for teams, shareable within a workspace, and governed by adm